In [1]:
spark.range(5).show()

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2,application_1782480617965_0003,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+

In [ ]:
import json
import urllib.parse
import urllib.request

from pyspark.sql import functions as F

BASE_URL = ""
TOKEN = ""

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
def fetch_batch_weather(station_id: str, limit: int = 500):
    params = urllib.parse.urlencode({
        "station_id": station_id,
        "limit": limit,
    })

    url = f"{BASE_URL}/weather/batch?{params}"

    request = urllib.request.Request(
        url,
        headers={"Authorization": f"Bearer {TOKEN}"},
        method="GET",
    )

    with urllib.request.urlopen(request, timeout=30) as response:
        data = json.loads(response.read().decode("utf-8"))

    return data["records"]


STATION_ID = "GDN_01"
LIMIT = 500

records = fetch_batch_weather(STATION_ID, LIMIT)

len(records), records[:2]

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

(500, [{'timestamp': '2026-06-26T14:07:56.754625+00:00', 'station_id': 'GDN_01', 'temperature': 19.08, 'humidity': 70.0, 'pressure': 1021.54, 'wind_speed': 8.69, 'wind_direction': 297, 'rain_mm': 0, 'cloud_cover': 96}, {'timestamp': '2026-06-26T13:57:56.754625+00:00', 'station_id': 'GDN_01', 'temperature': 19.98, 'humidity': 59.08, 'pressure': 1020.27, 'wind_speed': 0.85, 'wind_direction': 105, 'rain_mm': 0.0, 'cloud_cover': 41}])

In [4]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
)

schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("station_id", StringType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
    StructField("pressure", DoubleType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("wind_direction", IntegerType(), True),
    StructField("rain_mm", DoubleType(), True),
    StructField("cloud_cover", IntegerType(), True),
])

# Normalize numeric values before Spark sees them
normalized_records = []

for record in records:
    normalized_records.append({
        "timestamp": record.get("timestamp"),
        "station_id": record.get("station_id"),
        "temperature": float(record.get("temperature", 0)),
        "humidity": float(record.get("humidity", 0)),
        "pressure": float(record.get("pressure", 0)),
        "wind_speed": float(record.get("wind_speed", 0)),
        "wind_direction": int(record.get("wind_direction", 0)),
        "rain_mm": float(record.get("rain_mm", 0)),
        "cloud_cover": int(record.get("cloud_cover", 0)),
    })

raw_df = spark.createDataFrame(normalized_records, schema=schema)

raw_df.printSchema()
raw_df.show(5, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- timestamp: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- wind_direction: integer (nullable = true)
 |-- rain_mm: double (nullable = true)
 |-- cloud_cover: integer (nullable = true)

+--------------------------------+----------+-----------+--------+--------+----------+--------------+-------+-----------+
|timestamp                       |station_id|temperature|humidity|pressure|wind_speed|wind_direction|rain_mm|cloud_cover|
+--------------------------------+----------+-----------+--------+--------+----------+--------------+-------+-----------+
|2026-06-26T14:07:56.754625+00:00|GDN_01    |19.08      |70.0    |1021.54 |8.69      |297           |0.0    |96         |
|2026-06-26T13:57:56.754625+00:00|GDN_01    |19.98      |59.08   |1020.27 |0.85      |105           |0.0    |41         |
|2026-

In [5]:
def classify_weather_df(df):
    df = (
        df
        .withColumn("is_rainy", (F.col("rain_mm") > 0.5) & (F.col("cloud_cover") > 70))
        .withColumn("is_windy", F.col("wind_speed") > 10)
        .withColumn("is_cloudy", (F.col("cloud_cover") > 80) & (F.col("rain_mm") == 0))
        .withColumn("is_warm", F.col("temperature") > 22)
        .withColumn("is_cold", F.col("temperature") < 5)
        .withColumn("is_sunny", (F.col("cloud_cover") < 30) & (F.col("rain_mm") == 0))
        .withColumn(
            "is_storm_risk",
            (F.col("wind_speed") > 12)
            & (F.col("cloud_cover") > 75)
            & (F.col("pressure") < 1005),
        )
        .withColumn(
            "is_humid_hot",
            (F.col("humidity") > 85) & (F.col("temperature") > 24),
        )
    )

    tags_array = F.array_remove(
        F.array(
            F.when(F.col("is_storm_risk"), F.lit("storm-risk")),
            F.when(F.col("is_rainy"), F.lit("rainy")),
            F.when(F.col("is_windy"), F.lit("windy")),
            F.when(F.col("is_cloudy"), F.lit("cloudy")),
            F.when(F.col("is_sunny"), F.lit("sunny")),
            F.when(F.col("is_cold"), F.lit("cold")),
            F.when(F.col("is_warm"), F.lit("warm")),
            F.when(F.col("is_humid_hot"), F.lit("humid-hot")),
        ),
        None,
    )

    return (
        df
        .withColumn(
            "primary_weather_type",
            F.when(F.col("is_storm_risk"), "storm-risk")
            .when(F.col("is_rainy"), "rainy")
            .when(F.col("is_windy"), "windy")
            .when(F.col("is_cloudy"), "cloudy")
            .when(F.col("is_sunny"), "sunny")
            .when(F.col("is_cold"), "cold")
            .when(F.col("is_warm"), "warm")
            .when(F.col("is_humid_hot"), "humid-hot")
            .otherwise("normal"),
        )
        .withColumn(
            "weather_tags",
            F.when(F.size(tags_array) == 0, F.array(F.lit("normal"))).otherwise(tags_array),
        )
    )

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
classified_df = classify_weather_df(raw_df)

classified_df.select(
    "timestamp",
    "station_id",
    "temperature",
    "humidity",
    "pressure",
    "wind_speed",
    "rain_mm",
    "cloud_cover",
    "primary_weather_type",
    "weather_tags",
).show(20, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------------------+----------+-----------+--------+--------+----------+-------+-----------+--------------------+------------+
|timestamp                       |station_id|temperature|humidity|pressure|wind_speed|rain_mm|cloud_cover|primary_weather_type|weather_tags|
+--------------------------------+----------+-----------+--------+--------+----------+-------+-----------+--------------------+------------+
|2026-06-26T14:07:56.754625+00:00|GDN_01    |19.08      |70.0    |1021.54 |8.69      |0.0    |96         |cloudy              |null        |
|2026-06-26T13:57:56.754625+00:00|GDN_01    |19.98      |59.08   |1020.27 |0.85      |0.0    |41         |normal              |null        |
|2026-06-26T13:47:56.754625+00:00|GDN_01    |19.27      |87.31   |1008.9  |10.29     |2.16   |70         |windy               |null        |
|2026-06-26T13:37:56.754625+00:00|GDN_01    |14.93      |88.16   |1000.75 |5.29      |0.0    |50         |normal              |null        |
|2026-06-26T1

In [7]:
counts_df = (
    classified_df
    .groupBy("primary_weather_type")
    .count()
    .orderBy(F.desc("count"))
)

counts_df.show(truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-----+
|primary_weather_type|count|
+--------------------+-----+
|normal              |264  |
|windy               |105  |
|rainy               |64   |
|sunny               |41   |
|cloudy              |21   |
|storm-risk          |5    |
+--------------------+-----+

In [8]:
PROJECT_BUCKET = "weather-classification-tf-662832404229-2605"

classified_output_path = (
    f"s3://{PROJECT_BUCKET}/processed/weather_classified/station_id={STATION_ID}/"
)

counts_output_path = (
    f"s3://{PROJECT_BUCKET}/results/weather_type_counts/station_id={STATION_ID}/"
)

classified_df.write.mode("overwrite").parquet(classified_output_path)
counts_df.write.mode("overwrite").csv(counts_output_path, header=True)

classified_output_path, counts_output_path

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

('s3://weather-classification-tf-662832404229-2605/processed/weather_classified/station_id=GDN_01/', 's3://weather-classification-tf-662832404229-2605/results/weather_type_counts/station_id=GDN_01/')